# Week 10: Multi-Agent Systems

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week10-multi-agent.html

**How to use:** run the cells from the top (Runtime → Run all, or Shift+Enter cell by cell). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need a key: add it with the 🔑 Secrets panel on the left, or set `os.environ["OPENAI_API_KEY"] = "..."` in a cell. Examples that use Ollama need a local Ollama server and will not work in Colab. For GPU examples choose Runtime → Change runtime type → T4 GPU.

In [ ]:
# Libraries this lesson uses that are not preinstalled in Colab
%pip install -q mcp

In [ ]:
# Imports that some of the lesson's code fragments assume
import numpy as np

## Sample MCP server implementation (Python)

This code is a file of an application, not a notebook step. The cell saves it as `mcp_server.py`; run it from a terminal on your computer (or with `!python mcp_server.py` here, if it is a script).

In [ ]:
%%writefile mcp_server.py
from mcp.server import MCPServer   # mcp 1.x: from mcp.server.fastmcp import FastMCP as MCPServer

# Create a simple MCP server
mcp = MCPServer("FileReader", instructions="Reads files from the local filesystem")

# Define and implement a tool (the schema is derived from the signature)
@mcp.tool()
def read_file(path: str) -> str:
    """Reads content from a file."""
    try:
        with open(path, "r") as f:
            return f.read()
    except Exception as e:
        return f"error: {e}"

# Start the server (stdio transport by default)
if __name__ == "__main__":
    mcp.run()

## Run in your browser

**Writer-Critic loop: the evaluator-optimizer pattern (pure Python)**

In [ ]:
def writer(topic, notes=None):
    draft = f"{topic} replaces the feed-forward layer with several experts."
    if notes:
        draft += " A router picks the top-2 experts per token (Shazeer et al., 2017)."
    return draft

def critic(draft):
    score = 5 + 3 * ("(" in draft) + 2 * ("router" in draft)
    notes = [] if score >= 8 else ["add how routing works", "cite a source"]
    return score, notes

topic, notes = "Mixture of Experts", None
for round_ in range(1, 4):
    draft = writer(topic, notes)
    score, notes = critic(draft)
    print(f"round {round_}: score {score}/10  notes={notes}\n  {draft}")
    if score >= 8:
        print("accepted")
        break

## Test code for the course environment

**Minimal ReAct loop with a toy tool set (no API key needed: a scripted "LLM")**

In [ ]:
# 1. Minimal ReAct loop with a toy tool set (no API key needed: a scripted "LLM")
import re, json

TOOLS = {
    "multiply": lambda a, b: a * b,
    "search": lambda q: f"Stub result for '{q}'",
}

def fake_llm(trace):
    # A scripted policy standing in for the model; replace with a real LLM call.
    if "Observation" not in trace:
        return 'Thought: I need 17*23.\nAction: {"name":"multiply","arguments":{"a":17,"b":23}}'
    return "Thought: I have the answer.\nFinal Answer: 391"

trace = "Question: What is 17 times 23?"
for step in range(5):
    out = fake_llm(trace)
    trace += "\n" + out
    if "Final Answer:" in out:
        break
    call = json.loads(re.search(r"Action: (\{.*\})", out).group(1))
    obs = TOOLS[call["name"]](**call["arguments"])
    trace += f"\nObservation: {obs}"
print(trace)

**Short-term memory: a fixed-size context window with summarization fallback**

In [ ]:
# 2. Short-term memory: a fixed-size context window with summarization fallback
from collections import deque

WINDOW = 6  # tokens for illustration
history = deque(maxlen=WINDOW)
summary = ""

def add_turn(text):
    global summary
    for tok in text.split():
        if len(history) == WINDOW:
            summary = (summary + " " + history[0])[-40:]  # crude "summarizer"
        history.append(tok)

for turn in ["my name is Zaid", "I teach big data", "what is my name"]:
    add_turn(turn)
print("window :", list(history))
print("summary:", summary.strip())

**Long-term memory via RAG with a tiny vector store (numpy only)**

In [ ]:
# 3. Long-term memory via RAG with a tiny vector store (numpy only)
import numpy as np

docs = ["The meeting is on Tuesday at 10am.",
        "The project deadline is 30 November.",
        "Zaid prefers Python for data science labs."]

def embed(text):  # bag-of-characters embedding, enough to demonstrate retrieval
    v = np.zeros(128)
    for ch in text.lower():
        v[ord(ch) % 128] += 1
    return v / (np.linalg.norm(v) + 1e-9)

store = np.stack([embed(d) for d in docs])
query = "when is the deadline?"
scores = store @ embed(query)
print("retrieved:", docs[int(scores.argmax())])

**Supervisor routing in a multi-agent system**

In [ ]:
# 4. Supervisor routing in a multi-agent system
import re

AGENTS = {
    "math":   lambda task: f"[math agent] {eval(task)}",
    "writer": lambda task: f"[writer agent] Draft: {task.title()}",
}

def supervisor(task):
    # route to the math agent only if the whole task is an arithmetic expression
    is_math = re.fullmatch(r"[\d\s+\-*/().]+", task) is not None
    return AGENTS["math" if is_math else "writer"](task)

for t in ["12*12", "introduce multi-agent systems"]:
    print(supervisor(t))

**Parse an Agent Card and check whether it advertises a needed skill**

In [ ]:
# 1. Parse an Agent Card and check whether it advertises a needed skill
import json

card = json.loads("""{
  "name": "Recruitment Assistant", "version": "1.0.0",
  "url": "https://recruitment-agent.example.com/a2a",
  "capabilities": {"streaming": true, "pushNotifications": false},
  "defaultInputModes": ["text/plain", "application/json"],
  "defaultOutputModes": ["application/json"],
  "skills": [{"id": "find-candidates", "name": "findCandidates",
              "description": "Find candidates matching job requirements",
              "tags": ["recruiting", "search"]}]
}""")

def has_skill(card, name):
    return any(s["name"] == name or s["id"] == name for s in card["skills"])

print(card["name"], "supports findCandidates:", has_skill(card, "findCandidates"))
print("streams:", card["capabilities"]["streaming"])

**Task lifecycle state machine**

In [ ]:
# 2. Task lifecycle state machine
TRANSITIONS = {
    "submitted": {"working", "auth-required", "rejected", "canceled"},
    "working": {"input-required", "auth-required", "completed", "failed", "canceled"},
    "input-required": {"working", "canceled"},   # interrupted: waiting on the client
    "auth-required": {"working", "canceled"},    # interrupted: waiting on credentials
}
TERMINAL = {"completed", "failed", "canceled", "rejected"}

class Task:
    def __init__(self):
        self.state = "submitted"
        self.history = [self.state]
    def move(self, new):
        if self.state in TERMINAL or new not in TRANSITIONS[self.state]:
            raise ValueError(f"illegal {self.state} -> {new}")
        self.state = new
        self.history.append(new)

t = Task()
for s in ["working", "input-required", "working", "completed"]:
    t.move(s)
print(" -> ".join(t.history))

**Build a JSON-RPC 2.0 'message/send' request with a multi-part message**

In [ ]:
# 3. Build a JSON-RPC 2.0 'message/send' request with a multi-part message
import json, uuid

def message_send(text, data):
    return {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {
            "message": {
                "messageId": str(uuid.uuid4()),
                "role": "user",
                "parts": [
                    {"kind": "text", "text": text},
                    {"kind": "data", "data": data},
                ],
            },
        },
    }

print(json.dumps(message_send("Find Python developers in Amman",
                            {"jobTitle": "Data Engineer", "skills": ["Python", "Spark"]}),
                 indent=2))

**Simulate SSE status updates from a long-running remote agent**

In [ ]:
# 4. Simulate SSE status updates from a long-running remote agent
import json, time

def remote_agent_stream():
    for i, state in enumerate(["working", "working", "working", "completed"]):
        time.sleep(0.2)
        event = {"kind": "status-update", "taskId": "task-1", "contextId": "ctx-1",
                 "status": {"state": state}, "final": state == "completed"}
        # each SSE chunk carries one JSON-RPC response whose result is a TaskStatusUpdateEvent
        yield "data: " + json.dumps({"jsonrpc": "2.0", "id": 1, "result": event}) + "\n\n"

for chunk in remote_agent_stream():
    print(chunk, end="")